# 01 — Exploratory Data Analysis

This notebook covers the full exploratory data analysis for the **MovieLens 25M** dataset.
We load and inspect the six raw CSV files, then examine the distribution of ratings,
user activity, item popularity, genre composition, and metadata coverage. We conclude
by constructing a **user-wise temporal train / validation / test split** and persisting
all processed tables to disk for use in the subsequent notebooks.

The insights gathered here directly justify the feature choices, the temporal split
strategy, and the relevance threshold used throughout the evaluation pipeline.

**Steps:**
Load CSVs → Dataset overview → Rating distribution → User activity →
Item popularity & sparsity → Genre analysis → Tag & genome coverage →
Temporal analysis → Train / val / test split → Save processed data.


In [1]:
import sys
sys.path.insert(0, "..")

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

from hybrid_recsys.pipeline.data import (
    load_raw_ratings, load_raw_movies, load_raw_tags,
    load_genome_scores, load_genome_tags,
    build_movies_table, save_processed,
)
from hybrid_recsys.pipeline.splits import temporal_split, save_splits

FIGURES_DIR = Path("../artifacts/figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


def save_fig(fig, name: str) -> None:
    fig.write_html(str(FIGURES_DIR / f"{name}.html"))
    try:
        fig.write_image(str(FIGURES_DIR / f"{name}.png"), width=1200, height=600, scale=2)
    except Exception:
        pass  # kaleido not installed — HTML only
    fig.show()


## 1. Load Raw CSVs

In [2]:
ratings       = load_raw_ratings()
movies        = load_raw_movies()
tags          = load_raw_tags()
genome_scores = load_genome_scores()
genome_tags   = load_genome_tags()

print(f"ratings:       {ratings.shape}")
print(f"movies:        {movies.shape}")
print(f"tags:          {tags.shape}")
print(f"genome_scores: {genome_scores.shape}")
print(f"genome_tags:   {genome_tags.shape}")


ratings:       (25000095, 4)
movies:        (62423, 3)
tags:          (1093360, 4)
genome_scores: (15584448, 3)
genome_tags:   (1128, 2)


### Schema, dtypes & missing values

A first look at column types, memory footprint, and where values are missing —
before computing any statistics.


In [3]:
print("=== ratings ===")
ratings.info(memory_usage="deep")
print("\n=== movies (head) ===")
display(movies.head())
print("=== tags (head) ===")
display(tags.head(3))
print("=== genome_tags (head) ===")
display(genome_tags.head(3))

print("\nMissing values per table:")
for _name, _df in [("ratings", ratings), ("movies", movies), ("tags", tags)]:
    print(f"  {_name:<8} {_df.isna().sum().sum():,} NaNs across {_df.shape[1]} columns")


=== ratings ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000095 entries, 0 to 25000094
Data columns (total 4 columns):
 #   Column     Dtype  
---  ------     -----  
 0   userId     int32  
 1   movieId    int32  
 2   rating     float32
 3   timestamp  int64  
dtypes: float32(1), int32(2), int64(1)
memory usage: 476.8 MB

=== movies (head) ===


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


=== tags (head) ===


,userId,movieId,tag,timestamp
0,3,260,classic,1439472355
1,3,260,sci-fi,1439472256
2,4,1732,dark comedy,1573943598


=== genome_tags (head) ===


,tagId,tag
0,1,007
1,2,007 (series)
2,3,18th century



Missing values per table:
  ratings  0 NaNs across 4 columns
  movies   0 NaNs across 3 columns
  tags     16 NaNs across 4 columns


## 2. Dataset Overview

In [4]:
print(f"Unique users:    {ratings['userId'].nunique():>10,}")
print(f"Unique movies:   {ratings['movieId'].nunique():>10,}")
print(f"Total ratings:   {len(ratings):>10,}")
print(f"Rating range:    {ratings['rating'].min()} – {ratings['rating'].max()}")
print(
    f"Timestamp range: "
    f"{pd.to_datetime(ratings['timestamp'].min(), unit='s').date()} → "
    f"{pd.to_datetime(ratings['timestamp'].max(), unit='s').date()}"
)
print()
display(ratings.head(3))
display(movies.head(3))

print("Rating value — descriptive statistics:")
display(ratings[["rating"]].describe().T)


Unique users:       162,541
Unique movies:       59,047
Total ratings:   25,000,095
Rating range:    0.5 – 5.0
Timestamp range: 1995-01-09 → 2019-11-21



,userId,movieId,rating,timestamp
0,1,296,5.0,1147880044
1,1,306,3.5,1147868817
2,1,307,5.0,1147868828


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance


Rating value — descriptive statistics:


,count,mean,std,min,25%,50%,75%,max
rating,25000095.0,3.533855,1.060744,0.5,3.0,3.5,4.0,5.0


## 3. Rating Distribution

MovieLens uses a half-star scale from 0.5 to 5.0. The distribution is left-skewed:
users tend to rate movies they chose to watch, introducing a **positivity bias**
that must be accounted for when setting the relevance threshold for ranking metrics.


In [5]:
rating_counts = ratings["rating"].value_counts().sort_index().reset_index()
rating_counts.columns = ["rating", "count"]

fig = px.bar(
    rating_counts, x="rating", y="count",
    title="Rating Distribution — MovieLens 25M",
    labels={"rating": "Rating", "count": "Number of Ratings"},
    color="count", color_continuous_scale="Blues", text_auto=True,
)
fig.update_layout(coloraxis_showscale=False, bargap=0.2)
save_fig(fig, "01_rating_distribution")


## 4. User Activity — Long Tail

The number of ratings per user follows a heavy-tailed distribution. Most users
have rated relatively few movies, while a small minority have rated thousands.
This sparsity is the primary challenge for collaborative filtering models.


In [6]:
user_counts = ratings.groupby("userId").size().reset_index(name="n_ratings")

fig = px.histogram(
    user_counts, x="n_ratings", nbins=100,
    title="Ratings per User (log y-scale)",
    labels={"n_ratings": "Ratings per User", "count": "Number of Users"},
    color_discrete_sequence=["#EF553B"],
    log_y=True,
)
save_fig(fig, "02_user_activity")

print("Ratings per user — summary statistics:")
print(user_counts["n_ratings"].describe().round(1).to_string())


Ratings per user — summary statistics:
count    162541.0
mean        153.8
std         268.0
min          20.0
25%          36.0
50%          71.0
75%         162.0
max       32202.0


## 5. Item Popularity & Matrix Sparsity

Ratings per movie also follow a power-law distribution. We additionally compute
the **interaction matrix sparsity**, which quantifies what fraction of (user, item)
pairs have no observed rating and motivates both content-based and hybrid approaches.


In [7]:
item_counts = ratings.groupby("movieId").size().reset_index(name="n_ratings")

fig = px.histogram(
    item_counts, x="n_ratings", nbins=100,
    title="Ratings per Movie (log y-scale)",
    labels={"n_ratings": "Ratings per Movie", "count": "Number of Movies"},
    color_discrete_sequence=["#00CC96"],
    log_y=True,
)
save_fig(fig, "03_item_popularity")

n_users  = ratings["userId"].nunique()
n_items  = ratings["movieId"].nunique()
sparsity = 1 - len(ratings) / (n_users * n_items)

print(f"Interaction matrix: {n_users:,} users × {n_items:,} items")
print(f"Sparsity:           {sparsity:.4%}")
print(f"Density:            {1 - sparsity:.4%}")
print()
print("Ratings per movie — summary statistics:")
print(item_counts["n_ratings"].describe().round(1).to_string())


Interaction matrix: 162,541 users × 59,047 items
Sparsity:           99.7395%
Density:            0.2605%

Ratings per movie — summary statistics:
count    59047.0
mean       423.4
std       2477.9
min          1.0
25%          2.0
50%          6.0
75%         36.0
max      81491.0


### Long-tail concentration (Pareto)

How concentrated are ratings on the most popular movies? The cumulative curve below
makes the sparsity tangible — a small fraction of titles absorbs most of the ratings,
which is precisely why content signals matter for the rest of the catalogue.


In [8]:
sorted_counts = np.sort(item_counts["n_ratings"].values)[::-1]
cum_share     = np.cumsum(sorted_counts) / sorted_counts.sum()
movie_share   = np.arange(1, len(cum_share) + 1) / len(cum_share) * 100

fig = px.area(
    x=movie_share, y=cum_share * 100,
    title="Cumulative Share of Ratings vs. Share of Movies (Long Tail)",
    labels={"x": "Top % of movies (most-rated first)", "y": "% of all ratings"},
)
save_fig(fig, "12_long_tail_pareto")

for pct in [1, 5, 10, 20]:
    share = cum_share[int(pct / 100 * len(cum_share)) - 1] * 100
    print(f"Top {pct:>2}% of movies  ->  {share:.1f}% of all ratings")


Top  1% of movies  ->  47.6% of all ratings
Top  5% of movies  ->  84.5% of all ratings
Top 10% of movies  ->  94.0% of all ratings
Top 20% of movies  ->  98.3% of all ratings


## 6. Genre Analysis

Each movie is labelled with one or more pipe-separated genres. Genre frequency
and diversity directly determine how discriminative the multi-hot genre block
will be in the content-based feature matrix.


In [9]:
genre_series = movies["genres"].str.split("|").explode()
genre_counts = (
    genre_series[genre_series != "(no genres listed)"]
    .value_counts()
    .reset_index()
)
genre_counts.columns = ["genre", "count"]

fig = px.bar(
    genre_counts, x="genre", y="count",
    title="Movie Count by Genre",
    labels={"genre": "Genre", "count": "Number of Movies"},
    color="count", color_continuous_scale="Teal",
)
fig.update_layout(xaxis_tickangle=-40, coloraxis_showscale=False)
save_fig(fig, "04_genre_frequency")

print(f"Distinct genres: {len(genre_counts)}")
print(genre_counts.to_string(index=False))


Distinct genres: 19
      genre  count
      Drama  25606
     Comedy  16870
   Thriller   8654
    Romance   7719
     Action   7348
     Horror   5989
Documentary   5605
      Crime   5319
  Adventure   4145
     Sci-Fi   3595
   Children   2935
  Animation   2929
    Mystery   2925
    Fantasy   2731
        War   1874
    Western   1399
    Musical   1054
  Film-Noir    353
       IMAX    195


## 7. Tag & Genome Coverage

Two metadata sources could augment the genre signal: **free-text user tags** and the
**tag genome** (a dense 1,128-dim relevance vector). The cell below reports the *actual*
coverage of each across the full movie catalogue. Note: only the free-text tags feed the
content model — the genome is profiled here for reference but is **not** used as a feature.


In [10]:
movies_with_tags   = tags["movieId"].nunique()
movies_with_genome = genome_scores["movieId"].nunique()
total_movies       = movies["movieId"].nunique()

coverage = pd.DataFrame({
    "Source":         ["Free-text tags", "Tag genome"],
    "Movies covered": [movies_with_tags, movies_with_genome],
    "Coverage %":     [
        round(100 * movies_with_tags   / total_movies, 1),
        round(100 * movies_with_genome / total_movies, 1),
    ],
})
display(coverage)

fig = px.bar(
    coverage, x="Source", y="Coverage %",
    title="Metadata Coverage across Movies",
    color="Source", text_auto=True,
    color_discrete_sequence=["#636EFA", "#FFA15A"],
)
fig.update_layout(showlegend=False, yaxis_range=[0, 100])
save_fig(fig, "05_tag_coverage")


,Source,Movies covered,Coverage %
0,Free-text tags,45251,72.5
1,Tag genome,13816,22.1


## 8. Temporal Analysis

Rating volume is not uniform over time. Visualising monthly activity confirms
that a **temporal split** is meaningful: recent user behaviour differs from
historical patterns, so a random split would leak future information into training.


In [11]:
ratings["date"] = (
    pd.to_datetime(ratings["timestamp"], unit="s")
    .dt.to_period("M")
    .dt.to_timestamp()
)
monthly = ratings.groupby("date").size().reset_index(name="n_ratings")

fig = px.line(
    monthly, x="date", y="n_ratings",
    title="Monthly Rating Volume — MovieLens 25M",
    labels={"date": "Date", "n_ratings": "Ratings per Month"},
)
fig.update_traces(line_color="#636EFA")
save_fig(fig, "06_temporal")


## 9. Train / Val / Test Split

We apply a **user-wise temporal split**: each user's ratings are sorted by timestamp
and divided 80 / 10 / 10 into train / validation / test. Users with fewer than
5 ratings total are excluded. This protocol ensures no future signal leaks into
training and produces a realistic held-out evaluation.


In [12]:
train, val, test = temporal_split(ratings.drop(columns=["date"]))

split_stats = pd.DataFrame({
    "Split":      ["Train", "Validation", "Test"],
    "Ratings":    [len(train),                len(val),                len(test)],
    "Users":      [train["userId"].nunique(), val["userId"].nunique(), test["userId"].nunique()],
    "Movies":     [train["movieId"].nunique(),val["movieId"].nunique(),test["movieId"].nunique()],
    "% of total": [
        round(100 * len(train) / len(ratings), 1),
        round(100 * len(val)   / len(ratings), 1),
        round(100 * len(test)  / len(ratings), 1),
    ],
})
display(split_stats)


,Split,Ratings,Users,Movies,% of total
0,Train,19936012,162541,51195,79.7
1,Validation,2431061,162541,38498,9.7
2,Test,2633022,162541,41366,10.5


### 9b. Split Characterisation

Counts alone don't tell us whether the splits are *comparable*. Here we check the
rating distribution, per-user activity, and — most importantly for a hybrid /
content-based system — how many **cold items** appear in validation and test (movies
with no ratings in train, which pure collaborative filtering cannot score). We also
verify the temporal split introduced **no leakage**.


In [13]:
def describe_split(name, df):
    rpu = df.groupby("userId").size()
    return {
        "Split":               name,
        "Ratings":             len(df),
        "Users":               df["userId"].nunique(),
        "Movies":              df["movieId"].nunique(),
        "Mean rating":         round(df["rating"].mean(), 3),
        "Std rating":          round(df["rating"].std(), 3),
        "Median ratings/user": int(rpu.median()),
    }

char = pd.DataFrame([
    describe_split("Train", train),
    describe_split("Validation", val),
    describe_split("Test", test),
]).set_index("Split")
display(char)

# Cold items: movies appearing in val/test but never seen in train.
train_items = set(train["movieId"].unique())
for name, df in [("Validation", val), ("Test", test)]:
    cold_mask = ~df["movieId"].isin(train_items)
    print(f"{name}: {cold_mask.sum():,} ratings on cold items "
          f"({100 * cold_mask.mean():.2f}%) across "
          f"{df.loc[cold_mask, 'movieId'].nunique():,} distinct cold movies")

# Leakage sanity check: each user's last train rating should precede their first
# test rating (the temporal split sorts by timestamp before slicing).
chk = (
    train.groupby("userId")["timestamp"].max().rename("train_max").to_frame()
    .join(test.groupby("userId")["timestamp"].min().rename("test_min"))
    .dropna()
)
violations = int((chk["train_max"] > chk["test_min"]).sum())
print(f"\nLeakage check — users with train_max > test_min: {violations:,} / {len(chk):,}")


,Ratings,Users,Movies,Mean rating,Std rating,Median ratings/user
Split,,,,,,
Train,19936012,162541,51195,3.549,1.061,56
Validation,2431061,162541,38498,3.457,1.054,7
Test,2633022,162541,41366,3.488,1.059,8


Validation: 4,685 ratings on cold items (0.19%) across 3,913 distinct cold movies
Test: 7,449 ratings on cold items (0.28%) across 5,396 distinct cold movies

Leakage check — users with train_max > test_min: 0 / 162,541


### Rating distribution across splits

If the temporal split is sound, the *shape* of the rating distribution should be
stable across train / val / test — only the volume changes, not user behaviour.


In [14]:
dist_rows = []
for name, df in [("Train", train), ("Validation", val), ("Test", test)]:
    props = df["rating"].value_counts(normalize=True).sort_index()
    for rating_val, prop in props.items():
        dist_rows.append({"Split": name, "Rating": rating_val, "Proportion": prop})

fig = px.bar(
    pd.DataFrame(dist_rows), x="Rating", y="Proportion", color="Split",
    barmode="group", title="Rating Distribution by Split",
)
save_fig(fig, "11_rating_dist_by_split")


## 10. Save Processed Data

In [15]:
movies_processed = build_movies_table(movies, tags)

save_processed(movies_processed, "movies")
save_processed(ratings.drop(columns=["date"], errors="ignore"), "ratings")
save_splits(train, val, test)

print("Saved to data/processed/:")
for name in ["movies", "ratings", "split_train", "split_val", "split_test"]:
    print(f"  {name}.parquet")


Saved to data/processed/:
  movies.parquet
  ratings.parquet
  split_train.parquet
  split_val.parquet
  split_test.parquet


## Conclusion

- The dataset contains **25M ratings** from **162K users** on **62K movies** on a 0.5–5.0 half-star scale.
- A clear **positivity bias** is present: the modal rating is 4.0; ratings ≤ 2.0 are rare. A relevance threshold of ≥ 4.0 for ranking metrics is therefore well-motivated.
- Both user activity and item popularity follow **power-law distributions**, yielding a matrix sparsity of ≈99.7% — content-based signals are essential for cold-item and sparse-user scenarios.
- **Drama, Comedy, and Thriller** dominate the genre space; the multi-hot genre vector will be a strong and interpretable content feature.
- **Free-text tags** cover ~72% of catalogued movies; the **tag genome** covers only ~22% (≈13.8K movies). The content representation therefore uses the broader free-text tags, not the genome.
- Rating volume grows over time, confirming that a **temporal split** is the correct evaluation protocol for this dataset.
- The user-wise 80/10/10 split is saved to `data/processed/` and will be loaded identically by every subsequent notebook and model.
